# DAI Mission — Proposal Template
**Data & AI in Economics | TU Dortmund**

This notebook is your team's mission proposal. Fill in every section before submission. Once approved, you will extend this same notebook into your final deliverable.

> **Team size:** 2–3 students  
> **Deliverable:** This Jupyter Notebook (proposal → final submission in one file)


## 1. Team

| Role | Name | Student ID |
|------|------|------------|
| Lead | Emre Isildar| |
| Member |Mohammadali Vahabi | |
| Member *(optional)* | | |


## 2. Mission Title & Research Question

**Title:** The Gender Pay Gap at Work: Does Being a Woman Causally Reduce Your Salary?

**Research question:**  
Does gender causally affect base salary among full-time employees, after controlling for job title, seniority, education, and age — and if so, how large is the unexplained pay gap?

**Why it matters:**  
The gender pay gap remains one of the most debated labour market inequalities in modern economies. While raw wage differences between men and women are well documented, it is unclear how much of this gap reflects genuine discrimination versus differences in occupation, experience, or education. Using individual-level salary data, this mission applies causal inference methods to isolate the direct effect of gender on pay providing evidence relevant to equal pay legislation, corporate HR policy, and broader debates about workplace fairness.

## 3. Data

**Source(s):**  Glassdoor Gender Pay Gap Dataset — Kaggle (nilimajauhari).
Access: https://www.kaggle.com/datasets/nilimajauhari/glassdoor-analyze-gender-pay-gap
Licence: CC0 Public Domain.

Gender Pay Gap Europe 2010–2021 — Kaggle (gianinamariapetrascu).
Access: https://www.kaggle.com/datasets/gianinamariapetrascu/gender-pay-gap-europe-2010-2021
Licence: CC0 Public Domain.

**Unit of observation:** Glassdoor: One row = one individual employee (gender, salary, job title, education, age)
Europe Pay Gap: One row = one country-year observation (average gender pay gap % for that country and year)

**Key variables:**

| Variable | Type | Role (feature / target / instrument / ...) | Description |
|----------|------|---------------------------------------------|-------------|
| Gender| categorical| Treatment| Name of the German city|
| BasePay| numerical| Target| Annual base salary USD (Glassdoor)|
|JobTitle | categorical| Confounder| Job role (Glassdoor)|
| Seniority| numerical| Confounder| Years of seniority (Glassdoor)|
| Education| Categorical| Confounder| Education level (Glassdoor)|
| Age| numerical| Confounder| Employee age (Glassdoor)|
| Year| numerical| Feature| Year 2010–2021 (Europe dataset)|

**Potential data quality issues:**  
Glassdoor dataset is not Germany-specific; it reflects general corporate salary patterns from mixed countries
Sample size is small (~1000 rows), which may limit statistical power in causal analysis
Possible measurement error — salary data may be self-reported or simulated


In [37]:
# Data loading & first inspection

import pandas as pd
import numpy as np

df_glassdoor = pd.read_csv('Glassdoor Gender Pay Gap.csv')
df_europe = pd.read_csv('pay_gap_Europe.csv')

print("=== GLASSDOOR DATASET ===")
print(df_glassdoor.head())
print(df_glassdoor.info())
print(df_glassdoor.describe())

print("\n=== EUROPE PAY GAP DATASET ===")
print(df_europe.head())
print(df_europe.info())
print(df_europe.describe())

=== GLASSDOOR DATASET ===
              JobTitle  Gender  Age  PerfEval Education            Dept  \
0     Graphic Designer  Female   18         5   College      Operations   
1    Software Engineer    Male   21         5   College      Management   
2  Warehouse Associate  Female   19         4       PhD  Administration   
3    Software Engineer    Male   20         5   Masters           Sales   
4     Graphic Designer    Male   26         5   Masters     Engineering   

   Seniority  BasePay  Bonus  
0          2    42363   9938  
1          5   108476  11128  
2          5    90208   9268  
3          4   108080  10154  
4          5    99464   9319  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   JobTitle   1000 non-null   object
 1   Gender     1000 non-null   object
 2   Age        1000 non-null   int64 
 3   PerfEval   1000 non-null   int64

## 4. Planned Methods

Your mission **must** apply at least one technique from **each** of the three blocks below. Tick the ones you plan to use and briefly justify the choice.

### 4a. Causal Inference
- [X] Causal graph / DAG (DoWhy)
- [ ] Backdoor adjustment
- [ ] Instrumental variable
- [X] Propensity score stratification
- [ ] Other: ___

*Justification:* Gender is not randomly assigned — age, education, job title, and seniority are confounders. We use a causal DAG to model these relationships and propensity score stratification to isolate the direct effect of gender on BasePay.

### 4b. Supervised Learning
- [X] Linear / Ridge / Lasso regression
- [ ] Logistic regression
- [ ] k-Nearest Neighbors
- [ ] Support Vector Machine
- [X] Decision Tree / Random Forest
- [ ] Neural network (regression or classification)
- [ ] Other: ___

*Justification:*BasePay is a continuous variable, making regression the natural choice. Linear regression provides interpretable coefficients for each feature including gender. Random Forest is added to capture non-linear relationships and compare predictive performance against the linear baseline.

### 4c. Unsupervised Learning / Generative Models
- [X] K-Means clustering
- [ ] Hierarchical clustering
- [ ] Variational autoencoder
- [ ] GAN
- [ ] Other: ___

*Justification:* K-Means is applied to the Europe Pay Gap dataset to group European countries by their gender pay gap profiles over time. This reveals which cluster Germany belongs to and whether high-gap and low-gap country groups show distinct patterns.

## 5. Evaluation Strategy

*How will you know if your mission succeeded? Describe:*

### Causal Inference (DoWhy — Propensity Score Stratification):

Metric: Average Treatment Effect (ATE) — average salary difference between male and female employees after controlling for confounders
Validation: DoWhy built-in refutation tests (random common cause, placebo treatment, data subset refutation)
Baseline: Raw mean salary difference between genders before any adjustment

### Supervised Learning (Linear Regression + Random Forest):

Metric: RMSE (Root Mean Squared Error) and R² score for both models
Validation: 80/20 train-test split, 5-fold cross-validation
Baseline: Predicting mean BasePay for everyone (no-feature baseline), compared against both models

### Unsupervised Learning (K-Means — Europe Pay Gap):

Metric: Silhouette Score to evaluate cluster quality
Validation: Elbow method to determine optimal number of clusters (k)
Baseline: No clustering baseline — we interpret cluster profiles and identify which cluster Germany belongs to

## 6. Work Plan

| Step | Owner | Description |
|------|-------|-------------|
| 1 | | Data collection & cleaning |
| 2 | | EDA |
| 3 | | Causal inference block |
| 4 | | Supervised learning block |
| 5 | | Unsupervised / generative block |
| 6 | | Synthesis & write-up |


---
## 7. Results *(complete for final submission)*


### 7a. Causal Inference

In [39]:
# Causal inference analysis



### 7b. Supervised Learning

In [40]:
# Supervised learning analysis

### 7c. Unsupervised / Generative

In [41]:
# Unsupervised / generative analysis

## 8. Discussion & Conclusion *(complete for final submission)*

*Synthesise findings across all three method blocks. What does each lens reveal that the others miss? What are the limitations of your analysis?*
